In [0]:
%run ./06-Medallion-Approach

In [0]:
class medallionTestSuite:
    def __init__(self):
        self.base_path='/Volumes/cat_spark_streaming/landing/datasets/json_files/'
        self.test_path='/Volumes/cat_spark_streaming/bronze/data/invoice-data-files/'
        self.archive_path='/Volumes/cat_spark_streaming/bronze/archive/'
        self.catalog_name="cat_spark_streaming"
        self.bronze_schema="bronze"        
        self.silver_schema="silver"
        self.bronze_table="bronze_invoice"
        self.silver_table="silver_invoice"

    def clean(self):
        dbutils.fs.rm(f'{self.test_path}',True)
        dbutils.fs.rm(f'{self.archive_path}',True)
        dbutils.fs.rm('/Volumes/cat_spark_streaming/bronze/data/checkpoint/',True)
        dbutils.fs.rm('/Volumes/cat_spark_streaming/silver/checkpoint/',True)
        print('Cleaned and Recreating directories: ')
        dbutils.fs.mkdirs(self.test_path)
        dbutils.fs.mkdirs(self.archive_path)
        dbutils.fs.mkdirs('/Volumes/cat_spark_streaming/bronze/data/checkpoint/')
        dbutils.fs.mkdirs('/Volumes/cat_spark_streaming/silver/checkpoint/')        
        spark.sql(f"drop table if exists {self.catalog_name}.{self.bronze_schema}.{self.bronze_table}")
        spark.sql(f"drop table if exists {self.catalog_name}.{self.silver_schema}.{self.silver_table}")

    def ingestFiles(self,itr):
        print(f"\tStarting Ingestion...", end= '') 
        dbutils.fs.cp(f'{self.base_path}invoices-{itr}.json', f'{self.test_path}')
        print('Completed')

    def assertResult(self,expected_count):
        actual_count=spark.sql(f'select count(*) from {self.catalog_name}.{self.silver_schema}.{self.silver_table}').collect()[0][0]
        assert actual_count==expected_count, f"\tTest failed! {actual_count} is not equal to {expected_count}"

    def waitForMicrobatch(self,sleep=30):
        import time
        print(f"\tWait for {sleep} seconds",end='')
        time.sleep(sleep)
        print("Done")

    def runStreamTest(self):
        self.clean()
        obj_b=bronze()
        obj_s=silver()
        obj_b.process()
        obj_s.process()

        print("Starting the first iteration:")        
        self.ingestFiles(1)
        self.waitForMicrobatch()
        self.waitForMicrobatch()
        self.assertResult(1253)
        print("Validation passed.\n")

        obj_b.stop()
        obj_s.stop()
    

In [0]:
objTs=medallionTestSuite()
objTs.runStreamTest()